In [ ]:
!pip install langchain_core langchain_classic langchain_google_genai

In [ ]:
!pip install -U langchain-huggingface sentence-transformers

In [ ]:
!pip install unstructured
!pip install unstructured[pdf]

In [ ]:
!pip install unstructured
!pip install pdfminer.six
!pip install pi-heif
!pip install unstructured-inference
!pip install pikepdf
!pip install pdf2image


In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install -U langchain-groq

In [ ]:
from unstructured.partition.pdf import partition_pdf

elements=partition_pdf(
    filename="/content/Body-Fluids-and-Circulation-1.pdf",
    strategy='hi_res',
    infer_table_structure=True
)

In [ ]:
for i, element in enumerate(elements[:20]):
    print(f"\nElement {i}")
    print(type(element).__name__)
    print(str(element)[:500])
    print("-"*50)

In [ ]:
data = []

for element in elements:
    data.append({
        "type": type(element).__name__,
        "text": str(element)
    })

data[:5]

In [ ]:
texts = []
tables = []

for element in elements:
    if type(element).__name__ == "Table":
        tables.append(str(element))
    else:
        texts.append(str(element))

print("Text Blocks:", len(texts))
print("Tables:", len(tables))

In [ ]:
print(tables[1])

In [ ]:
from collections import Counter

element_types = Counter(type(e).__name__ for e in elements)

print(element_types)

In [ ]:
# This extract the table, text and images as well above one only extract the table and the text

# elements = partition_pdf(
#     filename="/content/Body-Fluids-and-Circulation-1.pdf",
#     strategy="hi_res",
#     infer_table_structure=True,
#     extract_image_block_types=["Image"],
#     extract_image_block_to_payload=True
# )

In [ ]:
for e in elements:
    if type(e).__name__ == "Image":
        print(e.metadata)
        break

In [ ]:
texts=[]
tables=[]
images=[]

for e in elements:

    element_type = type(e).__name__

    if element_type == "Table":
        tables.append(e)

    elif element_type == "Image":
        images.append(e)

    else:
        texts.append(e)

print(len(texts))
print(len(tables))
print(len(images))

In [ ]:
for i in range(5):
    print(f"\n--- Text {i} ---")
    print(type(texts[i]).__name__)
    print(texts[i].text[:500])

In [ ]:
for i, table in enumerate(tables):
    print(f"\n--- Table {i} ---")
    print(table.text[:1000])

In [ ]:
print(images[0].metadata)

In [ ]:
print(vars(images[0].metadata))

In [ ]:
for key, value in vars(images[0].metadata).items():
    print(f"{key}: {value}")

In [ ]:
documents = []
for t in texts:
    if hasattr(t, "text") and t.text.strip():
        documents.append({
            "content": t.text,
            "type": "text",
            "page": getattr(t.metadata, "page_number", None)
        })
for table in tables:
    documents.append({
        "content": table.text,
        "type": "table",
        "page": getattr(table.metadata, "page_number", None)
    })

In [ ]:
# import os


# os.environ["GOOGLE_API_KEY"] = ""

from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import google.generativeai as genai
import os


for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(model.name)

In [ ]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/gemini-embedding-001"
# )

In [ ]:
# vector = embeddings.embed_query("What is blood circulation in warm blooded in the mammal ?")
# print(len(vector))

In [ ]:
text_docs = []

for element in texts:

    content = getattr(element, "text", "")

    if content and content.strip():

        text_docs.append({
            "content": content,
            "type": "text",
            "page": getattr(element.metadata, "page_number", None)
        })

print(len(text_docs))

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     temperature=0
# )

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
table_docs = []

for table in tables:

    prompt = f"""
    Summarize this table for retrieval purposes.

    Table:
    {table.text}

    Create a concise but information-rich summary.
    """

    summary = llm.invoke(prompt).content

    table_docs.append({
        "content": summary,
        "raw_table": table.text,
        "type": "table",
        "page": getattr(table.metadata, "page_number", None)
    })

In [ ]:
table_docs.append({
    "summary": summary,
    "raw_table": table.text
})

In [ ]:
import base64
images[0].metadata.to_dict()

In [ ]:
# img_base64 = images[0].metadata.image_base64

# with open("sample.jpg", "wb") as f:
#     f.write(base64.b64decode(img_base64))

## Vision Caption

In [ ]:
# from PIL import Image
# import google.generativeai as genai

# img = Image.open("sample.jpg")
# model1 = genai.GenerativeModel("gemini-3.5-pro-preview")

# response = model1.generate_content([
#     """
#     Describe this image in detail for a retrieval system.
#     Mention labels, diagrams, biological structures,
#     and educational content.
#     """,
#     img
# ])

# print(response.text)

In [ ]:
# image_docs = []

# for img_element in images:

#     try:

#         img_base64 = img_element.metadata.image_base64

#         image_bytes = base64.b64decode(img_base64)

#         temp_file = "temp.jpg"

#         with open(temp_file, "wb") as f:
#             f.write(image_bytes)

#         image = Image.open(temp_file)

#         caption = model.generate_content([
#             """
#             Generate a detailed educational caption.
#             """,
#             image
#         ]).text

#         image_docs.append({
#             "content": caption,
#             "type": "image",
#             "page": getattr(
#                 img_element.metadata,
#                 "page_number",
#                 None
#             )
#         })

#     except Exception as e:
#         print(e)

In [ ]:
text_docs = []

for e in texts:

    content = getattr(e, "text", "")

    if content and content.strip():

        text_docs.append({
            "content": content,
            "type": "text",
            "page": getattr(e.metadata, "page_number", None)
        })

print(len(text_docs))

In [ ]:
table_docs = []

for table in tables:

    table_docs.append({
        "content": table.text,
        "type": "table",
        "page": getattr(table.metadata, "page_number", None)
    })

print(len(table_docs))

In [ ]:
all_docs = []

all_docs.extend(text_docs)
all_docs.extend(table_docs)

print(len(all_docs))

In [ ]:
print(type(all_docs[0]["content"]))
print(all_docs[0]["content"])

### -------------------------------------------------------------------

In [ ]:
for i, doc in enumerate(all_docs):

    if not isinstance(doc["content"], str):

        print(f"Problem at index {i}")
        print(type(doc["content"]))
        print(doc["content"])
        break

In [ ]:
import pprint

pprint.pprint(all_docs[979])

In [ ]:
for doc in all_docs[:5]:
    print(type(doc["content"]))

In [ ]:
from langchain_core.documents import Document


In [ ]:
documents = []

for doc in all_docs:

    documents.append(
        Document(
            page_content=doc["content"],
            metadata={
                "type": doc["type"],
                "page": doc["page"]
            }
        )
    )

In [ ]:
print(type(all_docs))
print(type(all_docs[0]))

# Chunking

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=doc["content"],
        metadata={
            "type": doc["type"],
            "page": doc["page"]
        }
    )
    for doc in all_docs
]

In [ ]:
full_text = "\n".join(
    [
        e.text
        for e in texts
        if hasattr(e, "text")
        and e.text.strip()
    ]
)

In [ ]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=200
# )

# chunks = splitter.split_documents(documents)

# print("Chunks:", len(chunks))

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents(
    [full_text]
)

In [ ]:
lengths = [len(c.page_content) for c in chunks]

print("Avg:", sum(lengths)/len(lengths))

In [ ]:
print(chunks[0].page_content[:500])
print(chunks[0].metadata)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5"
)

In [ ]:
!pip install langchain_community faiss-cpu rank_bm25

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)
vectorstore.save_local("faiss_index")

In [ ]:
query = "What is the function of the blood?"

docs = vectorstore.similarity_search(
    query,
    k=5
)

for doc in docs:
    print(doc.page_content)
    print("="*100)

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(chunks)

bm25.k = 10

In [ ]:
results = bm25.invoke(
    "What is the function of blood?"
)

print(results[0].page_content)

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25,
        vector_retriever
    ],
    weights=[0.4, 0.6]
)

In [ ]:
docs = hybrid_retriever.invoke(
    "Explain blood circulation"
)

print(len(docs))

In [ ]:
query = "What is the lifespan of RBCs?"

docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(docs[:5]):
    print(f"\n--- Doc {i+1} ---")
    print(doc.page_content[:1000])

# evaluation

In [ ]:
retrieval_gt = {
    "Who discovered blood circulation?": [8],

    "What are the functions of blood?": [2, 3],

    "What is the lifespan of RBCs?": [3, 7],

    "Name the plasma proteins and their functions.": [3],

    "Which blood group is universal donor?": [13],

    "Which blood group is universal acceptor?": [13],

    "What is erythroblastosis fetalis?": [14, 15],

    "What is serum?": [3],

    "Difference between open and closed circulatory system?": [8, 9],

    "Which vessels carry blood from lungs to heart?": [12],

    "What is the function of platelets?": [7],

    "What are granulocytes?": [4, 5],

    "What are agranulocytes?": [6, 7]
}

In [ ]:
from collections import defaultdict
import numpy as np

In [ ]:
def precision_at_k(retrieved_pages, relevant_pages, k):
    retrieved_k = retrieved_pages[:k]

    hits = len(
        set(retrieved_k) &
        set(relevant_pages)
    )

    return hits / k

In [ ]:
#recall@k
def recall_at_k(retrieved_pages, relevant_pages, k):
    retrieved_k = retrieved_pages[:k]

    hits = len(
        set(retrieved_k) &
        set(relevant_pages)
    )

    return hits / len(relevant_pages)

In [ ]:
def f1_at_k(p, r):

    if p + r == 0:
        return 0

    return 2 * p * r / (p + r)

In [ ]:
#Reciprocal rank
def reciprocal_rank(
    retrieved_pages,
    relevant_pages
):

    for rank, page in enumerate(
        retrieved_pages,
        start=1
    ):
        if page in relevant_pages:
            return 1 / rank

    return 0

In [ ]:
#ndcg
def ndcg_at_k(
    retrieved_pages,
    relevant_pages,
    k
):

    dcg = 0

    for i, page in enumerate(
        retrieved_pages[:k],
        start=1
    ):

        rel = 1 if page in relevant_pages else 0

        dcg += rel / np.log2(i + 1)

    ideal_hits = min(
        len(relevant_pages),
        k
    )

    idcg = 0

    for i in range(
        1,
        ideal_hits + 1
    ):
        idcg += 1 / np.log2(i + 1)

    return dcg / idcg if idcg > 0 else 0

In [ ]:
results = []

for query, relevant_pages in retrieval_gt.items():

    docs = hybrid_retriever.invoke(query)

    retrieved_pages = [
        d.metadata.get("page")
        for d in docs
    ]

    p = precision_at_k(
        retrieved_pages,
        relevant_pages,
        5
    )

    r = recall_at_k(
        retrieved_pages,
        relevant_pages,
        5
    )

    f1 = f1_at_k(p, r)

    rr = reciprocal_rank(
        retrieved_pages,
        relevant_pages
    )

    ndcg = ndcg_at_k(
        retrieved_pages,
        relevant_pages,
        5
    )

    results.append({
        "query": query,
        "precision@5": p,
        "recall@5": r,
        "f1@5": f1,
        "rr": rr,
        "ndcg@5": ndcg
    })

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

display(df)

print("\nAverage Metrics\n")

print(
    "Precision@5:",
    round(df["precision@5"].mean(), 4)
)

print(
    "Recall@5:",
    round(df["recall@5"].mean(), 4)
)

print(
    "F1@5:",
    round(df["f1@5"].mean(), 4)
)

print(
    "MRR:",
    round(df["rr"].mean(), 4)
)

print(
    "nDCG@5:",
    round(df["ndcg@5"].mean(), 4)
)

# Reranking in RAG

When we retrieve documents for a query, we often get a large set of candidate passages.  
**Reranking** is the step where we reorder those candidates so the most relevant ones (the "needle") rise to the top before the LLM sees them.

---

## Types of Rerankers

- **Cross-Encoders (e.g., MS MARCO MiniLM)**  
  Encode query + document together → deep relevance score.  
  High precision, good for nuanced queries.  
  Slower, limited context (~512 tokens).  
  [SentenceTransformers Cross-Encoders](https://www.sbert.net/docs/pretrained-models/ce-msmarco.html)

- **Cohere Rerank (API)**  
  Commercial service trained on diverse QA datasets.  
  Strong accuracy, easy integration.  
  Paid, adds latency.  
  [Cohere Rerank Docs](https://docs.cohere.com/docs/rerank)

- **NVIDIA RerankQA Models**  
  Optimized for QA tasks.  
  High precision, enterprise-ready.  
  Smaller context windows (~512 tokens).  
  [NVIDIA NIM RerankQA](https://developer.nvidia.com/nim)

- **Qwen3-Reranker-4B**  
  Open-source, multilingual, 32k context.  
  State-of-the-art free reranker, great for long docs & code.  
  Larger model → slower inference.  
  [Qwen3 Reranker GitHub](https://github.com/QwenLM/Qwen)

---

## Recall vs Precision

- **Recall** = how many of the truly relevant documents are retrieved.  
- **Precision** = how many of the retrieved documents are actually relevant.  

Reranking does not increase recall (you still only have the top‑k candidates),  
but it **boosts precision** by pushing the best matches higher.

---

## Impact in Numbers

- **Before reranking**  
  - Accuracy ~65–70% (correct chunk often buried at rank 5–10)  
  - Latency low (retriever only)

- **After reranking**  
  - Accuracy jumps to ~85–90% (correct chunk usually in top 1–3)  
  - Latency adds ~50–200ms depending on model size

---

## Which to Use When

- Fast, English-only apps → Cross-Encoder MiniLM  
- Production pipelines with budget → Cohere Rerank  
- Enterprise QA with GPU infra → NVIDIA RerankQA  
- Free, multilingual, long-context RAG → Qwen3-Reranker-4B  

---

In short: reranking is the **precision booster** in RAG.  
It costs some latency, but the payoff in accuracy is significant — the LLM gets the right context, which means better answers.


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

In [ ]:
#test

docs = hybrid_retriever.invoke(query)

In [ ]:
# #scoring using the reranker
# “Given this query, how relevant is each document chunk?”
# It returns a score for each, so you can reorder them and pick the best ones to feed into your LLM.
pairs = [
    (query, doc.page_content)
    for doc in docs
]

scores = reranker.predict(pairs)

In [ ]:
print(scores)

In [ ]:
#sort:sorting and selecting the top‑scoring documents after reranking
ranked_docs = sorted(
    zip(scores, docs),
    key=lambda x: x[0],
    reverse=True
)

top_docs = [
    doc
    for score, doc in ranked_docs[:5]
]

In [ ]:
print(top_docs)

for d in top_docs:
    print(d.page_content[:300])
    print("*"*100)

In [ ]:
context = "\n\n".join(
    [doc.page_content for doc in top_docs]
)

print(context[:1000])

In [ ]:
query=input("enter the query")

In [ ]:
pairs = [
    (query, doc.page_content)
    for doc in docs
]

scores = reranker.predict(pairs)

for score, doc in sorted(
    zip(scores, docs),
    key=lambda x: x[0],
    reverse=True
):
    print("\nScore:", score)
    print(doc.page_content[:500])

In [ ]:
response = llm.invoke(prompt)

answer = response.content

print(answer)

## retrival function


In [ ]:
def retrieve_and_rerank(query, top_k=3):

    docs = hybrid_retriever.invoke(query)

    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    top_docs = [
        doc
        for score, doc in ranked_docs[:top_k]
    ]

    return top_docs

In [ ]:
def build_context(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
You are a biology assistant.

Answer ONLY from the provided context.

If the answer is not available in the context,
say:
'I could not find the answer in the document.'

Context:
{context}

Question:
{question}

Answer:
"""
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt | llm | parser

In [ ]:
def rag_chain(query):

    docs = retrieve_and_rerank(query)

    context = build_context(docs)

    answer = chain.invoke(
        {
            "context": context,
            "question": query
        }
    )

    return answer

In [ ]:
rag_chain(
    "Name the founder of blood circulation."
)

In [ ]:
lengths = [len(chunk.page_content) for chunk in chunks]

print("Min:", min(lengths))
print("Max:", max(lengths))
print("Avg:", sum(lengths)/len(lengths))

In [ ]:
!pip install rouge-score bert-score

In [ ]:
rag_chain(query)

In [ ]:
generator_gt = {
    "Who discovered blood circulation?": [
        "William Harvey discovered blood circulation."
    ],
    "What are the functions of blood?": [
        "Blood transports oxygen and carbon dioxide, nutrients, hormones, waste products, regulates temperature and pH, helps in clotting, and protects against diseases."
    ],
    "What is the lifespan of RBCs?": [
        "The lifespan of mammalian erythrocytes (RBCs) is about 120 days."
    ],
    "Name the plasma proteins and their functions.": [
        "Fibrinogens help in clotting, Globulins provide defense, Albumins maintain osmotic balance."
    ],
    "Which blood group is universal donor?": [
        "Blood group O is the universal donor."
    ],
    "Which blood group is universal acceptor?": [
        "Blood group AB is the universal acceptor."
    ],
    "What is erythroblastosis fetalis?": [
        "It is a condition in Rh-negative mothers carrying Rh-positive fetuses where maternal antibodies destroy fetal RBCs."
    ],
    "What is serum?": [
        "Serum is blood plasma without coagulation factors."
    ],
    "Difference between open and closed circulatory system?": [
        "Open system: blood flows through lacunae and sinuses, low pressure, less efficient. Closed system: blood confined to vessels, faster circulation, more efficient."
    ],
    "Which vessels carry blood from lungs to heart?": [
        "Pulmonary veins carry oxygenated blood from lungs to the left auricle of the heart."
    ]
}


In [ ]:
generated_answers = {}

for question in generator_gt.keys():

    generated_answers[question] = rag_chain(
        question
    )

In [ ]:
for q, ans in generated_answers.items():

    print("\nQUESTION:", q)
    print("ANSWER:", ans)
    print("="*100)

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

rouge_scores = []

for question in generator_gt:

    reference = " ".join(
        generator_gt[question]
    )

    prediction = generated_answers[
        question
    ]

    score = scorer.score(
        reference,
        prediction
    )

    rouge_scores.append(
        score["rougeL"].fmeasure
    )

print(
    "Average ROUGE-L:",
    sum(rouge_scores) /
    len(rouge_scores)
)

In [ ]:
from bert_score import score
references = [
    " ".join(v)
    for v in generator_gt.values()
]

predictions = [
    generated_answers[q]
    for q in generator_gt.keys()
]

In [ ]:
P, R, F1 = score(
    predictions,
    references,
    lang="en"
)

print(
    "Average BERTScore:",
    F1.mean().item()
)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from sentence_transformers import SentenceTransformer

sim_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [ ]:
similarities = []

for question in generator_gt:

    reference = " ".join(
        generator_gt[question]
    )

    prediction = generated_answers[
        question
    ]

    ref_emb = sim_model.encode(
        reference
    )

    pred_emb = sim_model.encode(
        prediction
    )

    sim = cosine_similarity(
        [ref_emb],
        [pred_emb]
    )[0][0]

    similarities.append(sim)

print(
    "Average Semantic Similarity:",
    sum(similarities) /
    len(similarities)
)

In [ ]:
import pandas as pd

results = []

for question in generator_gt:

    results.append({
        "Question": question,
        "Reference":
            " ".join(generator_gt[question]),
        "Generated":
            generated_answers[question]
    })

df = pd.DataFrame(results)

df.head()

In [ ]:
import pandas as pd

results = []

for question in generator_gt:

    results.append({
        "Question": question,
        "Reference":
            " ".join(generator_gt[question]),
        "Generated":
            generated_answers[question]
    })

df = pd.DataFrame(results)

df.head()